In [ ]:
#imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier
import joblib
import os


In [ ]:
#Lets load the data
df = pd.read_csv('../input/titanic/train.csv') 
df.head()

In [ ]:
class TitanicPreprocessor:
    def __init__(self):
        self.preprocessor = None
    
    def feature_engineering(self, df):
        data = df.copy()
        data = data.drop(['PassengerId', 'Ticket'], axis=1, errors='ignore')
        data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
        title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
        data['Title'] = data['Title'].map(title_mapping).fillna('Other')
        data['Deck'] = data['Cabin'].str[0].fillna('Unknown')
        data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
        data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
        data['FarePerPerson'] = data['Fare'] / data['FamilySize'].replace(0, 1)
        data = data.drop(['Name', 'Cabin'], axis=1, errors='ignore')
        return data
    
    def fit_transform(self, df):
        data = self.feature_engineering(df)
        if 'Survived' in data.columns:
            y = data['Survived']
            X = data.drop('Survived', axis=1)
        else:
            y = None
            X = data
        
        numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson']
        categorical_features = ['Sex', 'Embarked', 'Title', 'Deck']
        
        numeric_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ])
        
        self.preprocessor = ColumnTransformer([
            ('num', numeric_pipeline, numeric_features),
            ('cat', categorical_pipeline, categorical_features)
        ])
        
        X_transformed = self.preprocessor.fit_transform(X)
        numeric_names = numeric_features
        categorical_names = self.preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
        all_feature_names = numeric_names + list(categorical_names)
        X_transformed = pd.DataFrame(X_transformed, columns=all_feature_names, index=X.index)
        
        return X_transformed, y
    
    def transform(self, df):
        if self.preprocessor is None:
            raise ValueError("Preprocessor not fitted. Call fit_transform first.")
        data = self.feature_engineering(df)
        if 'Survived' in data.columns:
            y = data['Survived']
            X = data.drop('Survived', axis=1)
        else:
            y = None
            X = data
        X_transformed = self.preprocessor.transform(X)
        numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson']
        categorical_features = ['Sex', 'Embarked', 'Title', 'Deck']
        numeric_names = numeric_features
        categorical_names = self.preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
        all_feature_names = numeric_names + list(categorical_names)
        X_transformed = pd.DataFrame(X_transformed, columns=all_feature_names, index=X.index)
        return X_transformed, y

preprocessor = TitanicPreprocessor()
X, y = preprocessor.fit_transform(df)

In [ ]:
#Split data for a training run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
lr_grid_params = {
    'C': [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2],
    'solver': ['lbfgs', 'liblinear', 'saga'],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'max_iter': [2000, 3000],
    'class_weight': [None, 'balanced', {0: 1, 1: 1.2}]
}

rf_grid_params = {
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 5, 6, 7],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 3],
    'max_features': ['sqrt', 0.5],
    'bootstrap': [True],
    'criterion': ['entropy', 'gini'],
    'class_weight': ['balanced', None]
}

catboost_grid_params = {
    'iterations': [500, 1000],
    'depth': [4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.1],
    'l2_leaf_reg': [3, 5, 7],
    'verbose': [False]
}

RANDOM_SEED = 42


In [ ]:
def grid_search_training(X_train, y_train, X_test, y_test, seed=RANDOM_SEED):
    results = {}
    best_models = {}
    
    model_configs = {
        'Logistic Regression': {'model_class': LogisticRegression, 'params': lr_grid_params, 'cv_folds': 5},
        'Random Forest': {'model_class': RandomForestClassifier, 'params': rf_grid_params, 'cv_folds': 3},
        'CatBoost': {'model_class': CatBoostClassifier, 'params': catboost_grid_params, 'cv_folds': 3}
    }
    
    for model_name, config in model_configs.items():
        print(f"Training {model_name}...")
        try:
            if model_name == 'Logistic Regression':
                model = config['model_class'](random_state=seed, max_iter=3000)
                compatible_grids = []
                param_grid = config['params'].copy()
                
                grid1 = param_grid.copy()
                grid1['solver'] = ['liblinear']
                grid1['penalty'] = ['l1', 'l2']
                compatible_grids.append(grid1)
                
                grid2 = param_grid.copy()
                grid2['solver'] = ['lbfgs']
                grid2['penalty'] = ['l2']
                compatible_grids.append(grid2)
                
                grid3 = param_grid.copy()
                grid3['solver'] = ['saga']
                grid3['penalty'] = ['l1', 'l2', 'elasticnet']
                compatible_grids.append(grid3)
                
                best_subsearch_score = 0
                best_subsearch_estimator = None
                for grid in compatible_grids:
                    cv = StratifiedKFold(n_splits=config['cv_folds'], shuffle=True, random_state=seed)
                    grid_search = GridSearchCV(estimator=model, param_grid=grid, cv=cv, scoring='accuracy', n_jobs=-1)
                    grid_search.fit(X_train, y_train)
                    if grid_search.best_score_ > best_subsearch_score:
                        best_subsearch_score = grid_search.best_score_
                        best_subsearch_estimator = grid_search.best_estimator_
                
                best_estimator = best_subsearch_estimator
                best_score = best_subsearch_score
                
            elif model_name == 'Random Forest':
                model = config['model_class'](random_state=seed)
                cv = StratifiedKFold(n_splits=config['cv_folds'], shuffle=True, random_state=seed)
                grid_search = GridSearchCV(estimator=model, param_grid=config['params'], cv=cv, scoring='accuracy', n_jobs=-1)
                grid_search.fit(X_train, y_train)
                best_estimator = grid_search.best_estimator_
                best_score = grid_search.best_score_
                
            elif model_name == 'CatBoost':
                model = config['model_class'](random_seed=seed, verbose=False)
                params_with_seed = config['params'].copy()
                params_with_seed['random_seed'] = [seed]
                cv = StratifiedKFold(n_splits=config['cv_folds'], shuffle=True, random_state=seed)
                grid_search = GridSearchCV(estimator=model, param_grid=params_with_seed, cv=cv, scoring='accuracy', n_jobs=-1)
                grid_search.fit(X_train, y_train)
                best_estimator = grid_search.best_estimator_
                best_score = grid_search.best_score_
            
            y_pred = best_estimator.predict(X_test)
            test_accuracy = accuracy_score(y_test, y_pred)
            
            results[model_name] = {
                'cv_score': best_score,
                'test_accuracy': test_accuracy,
                'test_precision': precision_score(y_test, y_pred),
                'test_recall': recall_score(y_test, y_pred),
                'test_f1': f1_score(y_test, y_pred),
                'best_params': best_estimator.get_params()
            }
            best_models[model_name] = best_estimator
            
        except Exception as e:
            results[model_name] = {'error': str(e)}
    
    return results, best_models

In [ ]:
#Lets run it

results, best_models = grid_search_training(X_train, y_train, X_test, y_test)




In [ ]:
#Save off our best model
print(f"{'Model':<20} {'CV Score':<10} {'Test Acc':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("-" * 70)

comparison_data = []
for model_name in ['Logistic Regression', 'Random Forest', 'CatBoost']:
    if model_name in results and 'error' not in results[model_name]:
        result = results[model_name]
        comparison_data.append({
            'Model': model_name,
            'Test_Accuracy': result['test_accuracy']
        })
        
        print(f"{model_name:<20} "
              f"{result['cv_score']:<10.4f} "
              f"{result['test_accuracy']:<10.4f} "
              f"{result['test_precision']:<10.4f} "
              f"{result['test_recall']:<10.4f} "
              f"{result['test_f1']:<10.4f}")

comparison_data.sort(key=lambda x: x['Test_Accuracy'], reverse=True)
if comparison_data:
    print(f"\nBest Model: {comparison_data[0]['Model']} ({comparison_data[0]['Test_Accuracy']:.4f})")

In [ ]:
save_dir = '/kaggle/working/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for model_name, model in best_models.items():
    filename = f"{model_name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, os.path.join(save_dir, filename))

joblib.dump(preprocessor, os.path.join(save_dir, 'preprocessor.joblib'))
joblib.dump(results, os.path.join(save_dir, 'results.joblib'))

if comparison_data:
    best_model_name = comparison_data[0]['Model']
    print(f"Best model saved: {best_model_name.lower().replace(' ', '_')}.joblib")

In [ ]:
#Lets retrain on the entire dataset with our best model and save it off
if comparison_data:
    best_model_name = comparison_data[0]['Model']
    best_model = best_models[best_model_name]
    
    #get our params
    best_params = results[best_model_name]['best_params']
    
    #create our model with the params
    if best_model_name == 'Logistic Regression':
        final_model = LogisticRegression(**{k: v for k, v in best_params.items() if k != 'random_state'})
        final_model.set_params(random_state=RANDOM_SEED)
    elif best_model_name == 'Random Forest':
        final_model = RandomForestClassifier(**{k: v for k, v in best_params.items() if k != 'random_state'})
        final_model.set_params(random_state=RANDOM_SEED)
    elif best_model_name == 'CatBoost':
        final_model = CatBoostClassifier(**{k: v for k, v in best_params.items() if k != 'random_seed'})
        final_model.set_params(random_seed=RANDOM_SEED, verbose=False)
    
    #train on all data
    final_model.fit(X_train, y_train)
    
    # Test final model performance
    final_predictions = final_model.predict(X_test)
    final_accuracy = accuracy_score(y_test, final_predictions)
    
    print(f"Final {best_model_name} accuracy on full training set: {final_accuracy:.4f}")
    
    #Save for our new notebook that will create a run for submission
    #already have our save dir
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    joblib.dump(final_model, os.path.join(save_dir, 'final_model.joblib'))
    
    print(f"Saved final model: final_model.joblib")
    